# 04. Evaluar y visualizar modelos

Este notebook compara las predicciones locales generadas por el notebook 03 contra los `reference_claim` anotados de `test.csv`. METEOR, ROUGE-L y BERTScore miden similitud con la referencia; no validan la veracidad médica de una noticia.

## Requisitos

Ejecuta antes el notebook 03. Para METEOR y BERTScore, instala `pip install -e '.[analysis,models,dev]'` y descarga una vez los recursos de NLTK con `python -m nltk.downloader wordnet omw-1.4`.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').is_file() and (ROOT.parent / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Abre el notebook desde la raíz del repositorio o desde notebooks/.')

DATASET_NAME = 'claims_500'
FINAL_DIR = ROOT / 'artifacts' / 'sem1' / DATASET_NAME
from claim_extraction.workflow import EXPERIMENT_VARIANTS

VARIANT = 'epocas_10'  # debe coincidir con el notebook 03
RUN_NAME = EXPERIMENT_VARIANTS[VARIANT]['run_name']
RUN_DIR = ROOT / 'artifacts' / 'sem1' / 'runs' / RUN_NAME
PREDICTIONS_DIR = RUN_DIR / 'predictions'
EVALUATION_DIR = RUN_DIR / 'evaluation'
EXPECTED_MODEL_IDS = ['t5s_spanish', 'barto_spanish', 'mbart50_es', 'umt5_base_es']
EXPECTED_SPLIT_SIZES = {'train': 400, 'validation': 50, 'test': 50}

print({'dataset': DATASET_NAME, 'run_dir': str(RUN_DIR), 'predictions_dir': str(PREDICTIONS_DIR)})

In [ ]:
import json

import pandas as pd

from claim_extraction.workflow import evaluate_predictions, utc_now, validate_sem1_splits

SPLIT_PATHS = {name: FINAL_DIR / f'{name}.csv' for name in ('train', 'validation', 'test')}
splits = validate_sem1_splits(SPLIT_PATHS)
observed_split_sizes = {name: len(frame) for name, frame in splits.items()}
if observed_split_sizes != EXPECTED_SPLIT_SIZES:
    raise ValueError(
        f'Splits incompatibles con {DATASET_NAME}: {observed_split_sizes}. '
        'Evalúa únicamente la corrida producida con el conjunto de 500 claims.'
    )
test = splits['test']

manifest_path = RUN_DIR / 'run_manifest.json'
if not manifest_path.is_file():
    raise FileNotFoundError(f'Falta el manifiesto de la corrida: {manifest_path}')
run_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
run_models = run_manifest.get('models', {})
if not isinstance(run_models, dict) or not run_models:
    raise ValueError('El manifiesto no registra modelos para evaluar.')
MODEL_ORDER = [model_id for model_id in EXPECTED_MODEL_IDS if model_id in run_models]
if not MODEL_ORDER:
    raise ValueError('El manifiesto no contiene IDs de modelos reconocidos.')

# Se verifica que cada modelo de esta corrida cubra exactamente las noticias de test.
prediction_paths = [PREDICTIONS_DIR / f'{model_id}.csv' for model_id in MODEL_ORDER]
missing = [path for path in prediction_paths if not path.is_file()]
if missing:
    raise FileNotFoundError('Faltan predicciones: ' + ', '.join(str(path) for path in missing))

prediction_frames = [pd.read_csv(path, dtype=str, keep_default_na=False) for path in prediction_paths]
predictions = pd.concat(prediction_frames, ignore_index=True)
expected_test_ids = set(test['record_id'])
for model_id, group in predictions.groupby('model_id', sort=True):
    if set(group['record_id']) != expected_test_ids or len(group) != len(test):
        raise ValueError(f'{model_id} no cubre exactamente los casos de test.csv.')

print({'test_rows': len(test), 'prediction_rows': len(predictions), 'models': sorted(predictions['model_id'].unique())})

## Calcular métricas

Cada salida se guarda localmente bajo la carpeta de la corrida. Los cuerpos de noticias, claims de referencia y predicciones individuales permanecen ignorados por Git.

In [ ]:
# Estas métricas comparan texto generado y referencia; no determinan veracidad médica.
summary, per_prediction = evaluate_predictions(predictions, test)
summary = summary.set_index('model_id').reindex(MODEL_ORDER).reset_index()
if summary[['meteor', 'rouge_l_f1', 'bertscore_f1']].isna().any().any():
    raise AssertionError('No se calcularon las tres métricas de referencia para todos los modelos.')

EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(EVALUATION_DIR / 'summary.csv', index=False, encoding='utf-8')
per_prediction.to_csv(EVALUATION_DIR / 'per_prediction.csv', index=False, encoding='utf-8')
for model_id, group in per_prediction.groupby('model_id', sort=True):
    model_dir = EVALUATION_DIR / model_id
    model_dir.mkdir(parents=True, exist_ok=True)
    summary.loc[summary['model_id'].eq(model_id)].to_csv(model_dir / 'summary.csv', index=False, encoding='utf-8')
    group.to_csv(model_dir / 'per_prediction.csv', index=False, encoding='utf-8')

status_path = RUN_DIR / 'run_status.json'
status = json.loads(status_path.read_text(encoding='utf-8')) if status_path.is_file() else {}
status['status'] = 'completed'
status['evaluated_at_utc'] = utc_now()
status_path.write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding='utf-8')

summary[['model_display_name', 'prediction_n', 'reference_n', 'meteor', 'rouge_l_f1', 'bertscore_f1']]

## Distribución de longitudes

La distribución se calcula sobre los 500 registros usados en la corrida. La longitud de noticia combina `title` y `body`; la de referencia corresponde al `reference_claim` anotado.

In [ ]:
# Esta figura describe el tamaño del material de entrada y de las afirmaciones, no la calidad del modelo.
import re

import matplotlib.pyplot as plt

TABLES_DIR = EVALUATION_DIR / 'tables'
FIGURES_DIR = EVALUATION_DIR / 'figures'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

all_records = pd.concat([splits[name] for name in ('train', 'validation', 'test')], ignore_index=True)
word_count = lambda value: len(re.findall(r'\b\w+\b', str(value), flags=re.UNICODE))
article_text = all_records['title'].fillna('') + ' ' + all_records['body'].fillna('')
lengths = pd.DataFrame({
    'record_id': all_records['record_id'],
    'news_words_title_body': article_text.map(word_count),
    'reference_claim_words': all_records['reference_claim'].fillna('').map(word_count),
})
if len(lengths) != sum(EXPECTED_SPLIT_SIZES.values()):
    raise AssertionError('La distribución debe incluir exactamente los 500 registros del conjunto final.')
lengths.to_csv(EVALUATION_DIR / 'article_and_claim_lengths.csv', index=False, encoding='utf-8')

figure, axes = plt.subplots(2, 1, figsize=(8, 7), layout='constrained')
for axis, column, title, color in [
    (axes[0], 'news_words_title_body', 'Noticias (título y cuerpo)', '#245b78'),
    (axes[1], 'reference_claim_words', 'Afirmaciones de referencia', '#27877c'),
]:
    median = lengths[column].median()
    axis.hist(lengths[column], bins=20, color=color, edgecolor='white')
    axis.axvline(median, color='#b75a3c', linestyle='--', label=f'Mediana: {median:.0f}')
    axis.set(title=title, xlabel='Longitud (palabras)', ylabel='Frecuencia')
    axis.legend(frameon=False)
    axis.grid(axis='y', alpha=.2)
figure.savefig(FIGURES_DIR / '02_distribucion_longitudes.png', dpi=180, bbox_inches='tight')
plt.show()

lengths.describe().T[['count', 'mean', '50%', 'min', 'max']]

## Cantidad de tokens

Cada modelo divide el texto con su propio tokenizador, así que el mismo texto ocupa una cantidad distinta de tokens en cada uno. Las figuras cuentan los tokens sin truncar sobre todos los registros del conjunto final y marcan el límite usado en el entrenamiento: lo que lo supera no llega al modelo. El CSV guarda solo conteos, sin texto.


In [ ]:
# Se cuenta sin truncar para ver cuánto texto queda fuera del límite de entrenamiento.
from claim_extraction.models import load_tokenizer
from claim_extraction.registry import get_model
from claim_extraction.text import input_for_model

input_texts = [input_for_model(row) for row in all_records.to_dict(orient='records')]
target_texts = all_records['reference_claim'].map(str).tolist()

token_rows = []
token_limits = {}
for model_id in MODEL_ORDER:
    spec = get_model(model_id)
    tokenizer = load_tokenizer(spec.base_checkpoint, model_id=model_id)
    token_limits[model_id] = {
        'source_tokens': int(spec.training['max_source_length']),
        'target_tokens': int(spec.training['max_target_length']),
    }
    source_ids = tokenizer(input_texts, truncation=False, verbose=False)['input_ids']
    target_ids = tokenizer(text_target=target_texts, truncation=False, verbose=False)['input_ids']
    for record_id, source, target in zip(all_records['record_id'], source_ids, target_ids, strict=True):
        token_rows.append({'record_id': record_id, 'model_id': model_id, 'source_tokens': len(source), 'target_tokens': len(target)})

token_counts = pd.DataFrame(token_rows)
token_counts.to_csv(EVALUATION_DIR / 'token_counts_by_model.csv', index=False, encoding='utf-8')

model_names = {model_id: get_model(model_id).display_name for model_id in MODEL_ORDER}

def plot_token_histograms(column, title, color, filename):
    figure, axes = plt.subplots(2, 2, figsize=(12, 7), layout='constrained', sharex=True, sharey=True)
    for axis, model_id in zip(axes.flat, MODEL_ORDER, strict=True):
        counts = token_counts.loc[token_counts['model_id'].eq(model_id), column]
        limit = token_limits[model_id][column]
        axis.hist(counts, bins=30, color=color, edgecolor='white')
        axis.axvline(counts.median(), color='#777777', linestyle=':', label=f'Mediana: {counts.median():.0f}')
        axis.axvline(limit, color='#b75a3c', linestyle='--', label=f'Límite: {limit} ({counts.gt(limit).mean():.1%} lo supera)')
        axis.set(title=model_names[model_id], xlabel='Tokens', ylabel='Noticias')
        axis.grid(axis='y', alpha=.2)
        axis.legend(fontsize=9, frameon=False)
    figure.suptitle(title)
    figure.savefig(FIGURES_DIR / filename, dpi=180, bbox_inches='tight')
    plt.show()

plot_token_histograms('source_tokens', 'Tokens de entrada (título y cuerpo)', '#245b78', '04_tokens_noticias.png')
plot_token_histograms('target_tokens', 'Tokens de las afirmaciones de referencia', '#27877c', '05_tokens_afirmaciones.png')

token_summary = pd.DataFrame([
    {
        'Modelo': model_names[model_id],
        'Texto': label,
        'Mediana': group[column].median(),
        'Percentil 95': group[column].quantile(.95),
        'Máximo': group[column].max(),
        'Límite': token_limits[model_id][column],
        'Superan el límite (%)': round(100 * group[column].gt(token_limits[model_id][column]).mean(), 1),
    }
    for model_id, group in token_counts.groupby('model_id', sort=False)
    for column, label in [('source_tokens', 'Noticia'), ('target_tokens', 'Afirmación')]
])
token_summary.to_csv(TABLES_DIR / '05_resumen_tokens.csv', index=False, encoding='utf-8')
token_summary


## Curvas de entrenamiento y validación

Se leen los historiales producidos por `Seq2SeqTrainer`. La línea vertical marca el checkpoint seleccionado por menor `eval_loss` dentro de cada arquitectura; no compara pérdidas entre arquitecturas.

In [ ]:
# El último trainer_state contiene el historial completo de la corrida de cada modelo.
curve_rows = []
selected_epochs = {}
for model_id in MODEL_ORDER:
    states = sorted((RUN_DIR / 'models' / model_id).glob('checkpoint-*/trainer_state.json'), key=lambda path: int(path.parent.name.split('-')[-1]))
    if not states:
        raise FileNotFoundError(f'No se encontró trainer_state.json para {model_id}.')
    state = json.loads(states[-1].read_text(encoding='utf-8'))
    display_name = run_models[model_id]['display_name']
    best_metric = state.get('best_metric')
    for entry in state.get('log_history', []):
        epoch = entry.get('epoch')
        step = entry.get('step')
        if epoch is None or step is None:
            continue
        if 'loss' in entry:
            curve_rows.append({'model_id': model_id, 'model_display_name': display_name, 'series': 'training_loss', 'epoch': epoch, 'step': step, 'loss': entry['loss']})
        if 'eval_loss' in entry:
            curve_rows.append({'model_id': model_id, 'model_display_name': display_name, 'series': 'validation_loss', 'epoch': epoch, 'step': step, 'loss': entry['eval_loss']})
            if best_metric is not None and abs(entry['eval_loss'] - best_metric) < 1e-10:
                selected_epochs[model_id] = epoch

curves = pd.DataFrame(curve_rows)
if curves.empty or set(curves['model_id']) != set(MODEL_ORDER):
    raise AssertionError('Faltan curvas de entrenamiento o validación para uno o más modelos.')
if set(selected_epochs) != set(MODEL_ORDER):
    raise AssertionError('No se pudo identificar el checkpoint seleccionado para todos los modelos.')
curves.to_csv(EVALUATION_DIR / 'training_curves.csv', index=False, encoding='utf-8')

figure, axes = plt.subplots(2, 2, figsize=(12, 8), layout='constrained')
for axis, model_id in zip(axes.flat, MODEL_ORDER, strict=True):
    model_curves = curves[curves['model_id'].eq(model_id)]
    training = model_curves[model_curves['series'].eq('training_loss')]
    validation = model_curves[model_curves['series'].eq('validation_loss')]
    axis.plot(training['epoch'], training['loss'], label='Entrenamiento', color='#245b78')
    axis.plot(validation['epoch'], validation['loss'], label='Validación', marker='o', color='#c77c35')
    selected_epoch = selected_epochs[model_id]
    axis.axvline(selected_epoch, color='#777777', linestyle=':', label=f'Checkpoint elegido (época {selected_epoch:g})')
    axis.set(title=run_models[model_id]['display_name'], xlabel='Época', ylabel='Pérdida')
    axis.grid(alpha=.2)
    axis.legend(fontsize=8, frameon=False)
figure.savefig(FIGURES_DIR / '03_curvas_entrenamiento_validacion.png', dpi=180, bbox_inches='tight')
plt.show()

pd.DataFrame([{'model_id': model_id, 'checkpoint_epoch': epoch} for model_id, epoch in selected_epochs.items()])

## Tablas y figura locales

Estas tablas sirven para redactar resultados. No publiques textos, anotaciones ni predicciones individuales: solo los agregados revisados.

In [ ]:
split_table = pd.DataFrame([
    {'split': name, 'filas': len(frame)} for name, frame in splits.items()
])
metrics_table = summary[[
    'model_id', 'model_display_name', 'prediction_n', 'reference_n',
    'meteor', 'rouge_l_f1', 'bertscore_f1',
]].copy()
diagnostics_table = summary[[
    'model_id', 'empty_rate', 'shorter_than_5_tokens_rate',
    'mean_prediction_tokens_operational', 'duplicate_output_rate',
]].copy()
split_table.to_csv(TABLES_DIR / '01_distribucion_particiones.csv', index=False, encoding='utf-8')
metrics_table.to_csv(TABLES_DIR / '02_metricas_principales.csv', index=False, encoding='utf-8')
diagnostics_table.to_csv(TABLES_DIR / '03_diagnosticos_generacion.csv', index=False, encoding='utf-8')

# Tabla de presentación: describe las salidas generadas; no mide veracidad médica.
diagnostics_display = summary[[
    'model_display_name', 'prediction_n', 'empty_rate',
    'shorter_than_5_tokens_rate', 'mean_prediction_tokens_operational',
    'duplicate_output_rate',
]].copy().rename(columns={
    'model_display_name': 'Modelo',
    'prediction_n': 'Afirmaciones generadas (n)',
    'empty_rate': 'Salidas vacías (%)',
    'shorter_than_5_tokens_rate': 'Menos de 5 tokens (%)',
    'mean_prediction_tokens_operational': 'Longitud media (tokens léxicos)',
    'duplicate_output_rate': 'Repeticiones adicionales (%)',
})
for column in ('Salidas vacías (%)', 'Menos de 5 tokens (%)', 'Repeticiones adicionales (%)'):
    diagnostics_display[column] = (100 * diagnostics_display[column]).round(1)
diagnostics_display['Longitud media (tokens léxicos)'] = diagnostics_display['Longitud media (tokens léxicos)'].round(2)
diagnostics_display.to_csv(TABLES_DIR / '04_diagnosticos_afirmaciones_generadas.csv', index=False, encoding='utf-8')

metric_columns = {'meteor': 'METEOR', 'rouge_l_f1': 'ROUGE-L F1', 'bertscore_f1': 'BERTScore F1'}
figure, axes = plt.subplots(1, 3, figsize=(12, 3.8), layout='constrained')
for axis, (column, label) in zip(axes, metric_columns.items(), strict=True):
    bars = axis.bar(metrics_table['model_display_name'], metrics_table[column], color='#277f8e')
    axis.bar_label(bars, labels=[f'{value:.3f}' for value in metrics_table[column]], padding=3, fontsize=9)
    axis.set(title=label, ylabel='Puntaje medio', ylim=(0, 1))
    axis.tick_params(axis='x', rotation=20)
    axis.grid(axis='y', alpha=.2)
figure.savefig(FIGURES_DIR / '01_comparacion_metricas.png', dpi=180, bbox_inches='tight')
plt.show()

print(f'Resultados locales: {EVALUATION_DIR.relative_to(ROOT)}')

print('\nDiagnósticos de las afirmaciones generadas:')
diagnostics_display

## Comparar variantes

Reúne las variantes ya evaluadas en este notebook. La columna `elegida_por_validacion` marca, para cada modelo, la variante con menor pérdida de validación. Esa elección no usa el test: METEOR, ROUGE-L y BERTScore solo se reportan.


In [ ]:
# Evalúa cada variante cambiando VARIANT antes de ejecutar esta comparación.
runs_dir = ROOT / 'artifacts' / 'sem1' / 'runs'
comparison_rows = []
for variant, config in EXPERIMENT_VARIANTS.items():
    variant_dir = runs_dir / config['run_name']
    for model_id in EXPECTED_MODEL_IDS:
        summary_path = variant_dir / 'evaluation' / model_id / 'summary.csv'
        states = sorted((variant_dir / 'models' / model_id).glob('checkpoint-*/trainer_state.json'), key=lambda path: int(path.parent.name.split('-')[-1]))
        if not summary_path.is_file() or not states:
            continue
        history = json.loads(states[-1].read_text(encoding='utf-8'))['log_history']
        evaluations = [row for row in history if 'eval_loss' in row]
        best = min(evaluations, key=lambda row: row['eval_loss'])
        metrics = pd.read_csv(summary_path).iloc[0]
        comparison_rows.append({
            'modelo': metrics['model_display_name'], 'model_id': model_id, 'variante': variant,
            'epocas_ejecutadas': round(evaluations[-1]['epoch']), 'mejor_epoca': round(best['epoch']),
            'perdida_validacion': best['eval_loss'], 'meteor_test': metrics['meteor'],
            'rouge_l_f1_test': metrics['rouge_l_f1'], 'bertscore_f1_test': metrics['bertscore_f1'],
        })
if not comparison_rows:
    raise RuntimeError('Todavía no hay variantes evaluadas.')

comparison = pd.DataFrame(comparison_rows)
comparison['elegida_por_validacion'] = comparison.index.isin(comparison.groupby('model_id')['perdida_validacion'].idxmin())
comparison.to_csv(runs_dir / 'comparacion_variantes.csv', index=False, encoding='utf-8')
comparison
